# LoRA Fine-tuning from Scratch

> Implement Low-Rank Adaptation (LoRA) for efficient LLM fine-tuning

**LoRA Paper:** Hu et al., 2021 — *LoRA: Low-Rank Adaptation of Large Language Models*

**Key Idea:** Instead of updating full weight matrix W, learn low-rank decomposition:
```
W' = W + ΔW = W + BA
```
Where B (d×r) and A (r×k) are low-rank matrices with r << min(d,k).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple

## 1. LoRA Layer Implementation

In [ ]:
class LoRALayer:
    """LoRA: Low-Rank Adaptation layer"""
    
    def __init__(self, in_features: int, out_features: int, rank: int = 4, alpha: float = 1.0):
        """
        Args:
            in_features: Input dimension
            out_features: Output dimension
            rank: Low-rank dimension (r)
            alpha: Scaling factor (typically alpha = 2*rank)
        """
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        # Frozen pre-trained weights (not updated)
        self.W = np.random.randn(out_features, in_features) * 0.02
        
        # Trainable low-rank matrices
        # B initialized to zero (so ΔW starts at 0)
        self.B = np.zeros((out_features, rank))
        # A initialized with small random values
        self.A = np.random.randn(rank, in_features) * 0.01
        
        # Gradients
        self.dB = np.zeros_like(self.B)
        self.dA = np.zeros_like(self.A)
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        """
        Forward: h = x @ W^T + (x @ A^T @ B^T) * scaling
        """
        self.x = x  # Store for backward
        
        # Base model output (frozen)
        base_out = x @ self.W.T
        
        # LoRA adaptation
        lora_out = (x @ self.A.T @ self.B.T) * self.scaling
        
        return base_out + lora_out
    
    def backward(self, dout: np.ndarray) -> np.ndarray:
        """
        Backward pass for LoRA parameters
        """
        # Gradient w.r.t. B
        # dout shape: (batch, out_features)
        # x @ A.T shape: (batch, rank)
        self.dB = (dout.T @ (self.x @ self.A.T)) * self.scaling / self.x.shape[0]
        
        # Gradient w.r.t. A
        # B.T @ dout.T shape: (rank, batch)
        # self.x shape: (batch, in_features)
        self.dA = ((self.B.T @ dout.T) @ self.x) * self.scaling / self.x.shape[0]
        
        # Gradient w.r.t. input
        dx = dout @ self.W
        dx += (dout @ self.B @ self.A) * self.scaling
        
        return dx
    
    def parameters(self) -> dict:
        """Return trainable parameters"""
        return {'B': self.B, 'A': self.A}
    
    def num_trainable_params(self) -> int:
        """Count trainable parameters"""
        return self.B.size + self.A.size
    
    def num_frozen_params(self) -> int:
        """Count frozen parameters"""
        return self.W.size

# Test LoRA layer
lora = LoRALayer(in_features=512, out_features=512, rank=8, alpha=16)

print(f"LoRA Config: rank={lora.rank}, alpha={lora.alpha}, scaling={lora.scaling:.4f}")
print(f"Trainable params: {lora.num_trainable_params():,}")
print(f"Frozen params: {lora.num_frozen_params():,}")
print(f"Reduction: {lora.num_frozen_params() / lora.num_trainable_params():.1f}x fewer params")

# Forward test
x = np.random.randn(4, 512)
out = lora.forward(x)
print(f"\nInput shape: {x.shape}, Output shape: {out.shape}")

## 2. LoRA Applied to Attention Q, K, V, O Projections

In [ ]:
class LoRAAttention:
    """Multi-head attention with LoRA on all projections"""
    
    def __init__(self, d_model: int, n_heads: int, rank: int = 4, alpha: float = 1.0,
                 lora_targets: list = ['q', 'v']):
        """
        Args:
            lora_targets: Which projections to apply LoRA to
                         Common: ['q', 'v'] or ['q', 'k', 'v', 'o']
        """
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.lora_targets = lora_targets
        
        # Create projections (with or without LoRA)
        self.q_proj = LoRALayer(d_model, d_model, rank, alpha) if 'q' in lora_targets else None
        self.k_proj = LoRALayer(d_model, d_model, rank, alpha) if 'k' in lora_targets else None
        self.v_proj = LoRALayer(d_model, d_model, rank, alpha) if 'v' in lora_targets else None
        self.o_proj = LoRALayer(d_model, d_model, rank, alpha) if 'o' in lora_targets else None
        
        # Frozen projections for non-LoRA targets
        if self.q_proj is None:
            self.W_q = np.random.randn(d_model, d_model) * 0.02
        if self.k_proj is None:
            self.W_k = np.random.randn(d_model, d_model) * 0.02
        if self.v_proj is None:
            self.W_v = np.random.randn(d_model, d_model) * 0.02
        if self.o_proj is None:
            self.W_o = np.random.randn(d_model, d_model) * 0.02
        
        # Causal mask
        self.mask = np.tril(np.ones((100, 100)))  # Max seq len
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        B, T, C = x.shape
        
        # Q, K, V projections
        q = self.q_proj.forward(x) if self.q_proj else x @ self.W_q.T
        k = self.k_proj.forward(x) if self.k_proj else x @ self.W_k.T
        v = self.v_proj.forward(x) if self.v_proj else x @ self.W_v.T
        
        # Reshape for multi-head
        q = q.reshape(B, T, self.n_heads, self.head_dim).transpose(0, 2, 1, 3)
        k = k.reshape(B, T, self.n_heads, self.head_dim).transpose(0, 2, 1, 3)
        v = v.reshape(B, T, self.n_heads, self.head_dim).transpose(0, 2, 1, 3)
        
        # Attention
        scores = (q @ k.transpose(0, 1, 3, 2)) / np.sqrt(self.head_dim)
        scores = np.where(self.mask[:T, :T] == 1, scores, -1e9)
        
        attn = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
        attn = attn / np.sum(attn, axis=-1, keepdims=True)
        
        out = attn @ v
        out = out.transpose(0, 2, 1, 3).reshape(B, T, C)
        
        # Output projection
        out = self.o_proj.forward(out) if self.o_proj else out @ self.W_o.T
        
        return out
    
    def count_params(self) -> Tuple[int, int]:
        """Return (trainable, frozen) parameter counts"""
        trainable = 0
        frozen = 0
        
        for proj in [self.q_proj, self.k_proj, self.v_proj, self.o_proj]:
            if proj:
                trainable += proj.num_trainable_params()
                frozen += proj.num_frozen_params()
            else:
                frozen += self.d_model * self.d_model
        
        return trainable, frozen

# Test LoRA attention
lora_attn = LoRAAttention(d_model=256, n_heads=4, rank=8, alpha=16, lora_targets=['q', 'v'])
trainable, frozen = lora_attn.count_params()

print(f"LoRA Attention (q, v only):")
print(f"  Trainable: {trainable:,} params")
print(f"  Frozen: {frozen:,} params")
print(f"  Total: {trainable + frozen:,} params")
print(f"  Trainable %: {100*trainable/(trainable+frozen):.2f}%")

x = np.random.randn(2, 32, 256)
out = lora_attn.forward(x)
print(f"\nInput: {x.shape}, Output: {out.shape}")

## 3. Parameter Efficiency Comparison

In [ ]:
# Compare different LoRA configurations
configs = [
    {'name': 'Full Fine-tuning', 'targets': [], 'rank': 0},
    {'name': 'LoRA (q,v) r=4', 'targets': ['q', 'v'], 'rank': 4},
    {'name': 'LoRA (q,v) r=8', 'targets': ['q', 'v'], 'rank': 8},
    {'name': 'LoRA (q,v) r=16', 'targets': ['q', 'v'], 'rank': 16},
    {'name': 'LoRA (q,k,v,o) r=4', 'targets': ['q', 'k', 'v', 'o'], 'rank': 4},
    {'name': 'LoRA (q,k,v,o) r=8', 'targets': ['q', 'k', 'v', 'o'], 'rank': 8},
]

d_model = 512
n_heads = 8

results = []
for cfg in configs:
    if cfg['rank'] == 0:
        # Full fine-tuning: all params trainable
        total = 4 * d_model * d_model  # q, k, v, o
        trainable = total
        frozen = 0
    else:
        attn = LoRAAttention(d_model, n_heads, rank=cfg['rank'], alpha=cfg['rank']*2,
                            lora_targets=cfg['targets'])
        trainable, frozen = attn.count_params()
        total = trainable + frozen
    
    results.append({
        'name': cfg['name'],
        'trainable': trainable,
        'frozen': frozen,
        'total': total,
        'pct': 100 * trainable / total
    })

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = [r['name'] for r in results]
trainable = [r['trainable'] / 1000 for r in results]  # In thousands
frozen = [r['frozen'] / 1000 for r in results]

x = np.arange(len(names))
width = 0.35

axes[0].bar(x - width/2, trainable, width, label='Trainable', color='#FF6B6B')
axes[0].bar(x + width/2, frozen, width, label='Frozen', color='#4ECDC4')
axes[0].set_ylabel('Parameters (thousands)')
axes[0].set_title('Parameter Count Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=45, ha='right')
axes[0].legend()

# Percentage trainable
pct = [r['pct'] for r in results]
colors = ['#FF6B6B' if p == 100 else '#45B7D1' for p in pct]
axes[1].bar(names, pct, color=colors, edgecolor='black')
axes[1].set_ylabel('% Trainable')
axes[1].set_title('Trainable Parameter Percentage')
axes[1].set_xticklabels(names, rotation=45, ha='right')

for i, p in enumerate(pct):
    axes[1].text(i, p + 1, f'{p:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Training Simulation

In [ ]:
# Simulate fine-tuning task: sentiment classification
# We'll create a simple binary classification head on top of LoRA-adapted features

class SimpleClassifier:
    """Classifier with LoRA-adapted transformer"""
    
    def __init__(self, d_model=128, seq_len=16, rank=4):
        self.attn = LoRAAttention(d_model, n_heads=4, rank=rank, alpha=rank*2,
                                 lora_targets=['q', 'v'])
        self.classifier_W = np.random.randn(1, d_model) * 0.02
        self.classifier_b = np.zeros(1)
    
    def forward(self, x):
        # Transformer encoding
        h = self.attn.forward(x)
        # Pool (mean over sequence)
        pooled = np.mean(h, axis=1)
        # Classifier
        logits = pooled @ self.classifier_W.T + self.classifier_b
        return 1 / (1 + np.exp(-logits))  # Sigmoid

# Generate synthetic data
np.random.seed(42)
n_samples = 1000
d_model = 128
seq_len = 16

# Positive class: higher mean values
X_pos = np.random.randn(n_samples // 2, seq_len, d_model) + 0.5
X_neg = np.random.randn(n_samples // 2, seq_len, d_model) - 0.5
X = np.concatenate([X_pos, X_neg], axis=0)
y = np.concatenate([np.ones(n_samples // 2), np.zeros(n_samples // 2)]).reshape(-1, 1)

# Shuffle
perm = np.random.permutation(n_samples)
X, y = X[perm], y[perm]

# Split
split = int(0.8 * n_samples)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print(f"Train: {X_train.shape}, Val: {X_val.shape}")
print(f"Positive ratio: {y.mean():.2f}")

In [ ]:
# Training with LoRA
model = SimpleClassifier(d_model=128, seq_len=16, rank=8)

lr = 0.01
epochs = 50
batch_size = 32

train_losses = []
val_accs = []

for epoch in range(epochs):
    # Mini-batch training
    epoch_loss = 0
    for i in range(0, len(X_train), batch_size):
        xb = X_train[i:i+batch_size]
        yb = y_train[i:i+batch_size]
        
        # Forward
        pred = model.forward(xb)
        
        # BCE loss
        loss = -np.mean(yb * np.log(pred + 1e-8) + (1-yb) * np.log(1-pred + 1e-8))
        epoch_loss += loss
        
        # Simple gradient update (simplified)
        grad = pred - yb
        model.classifier_W -= lr * np.mean(grad * np.mean(xb, axis=1, keepdims=True).transpose(0, 2, 1), axis=0).T
    
    train_losses.append(epoch_loss / (len(X_train) // batch_size))
    
    # Validation accuracy
    val_pred = model.forward(X_val)
    val_acc = np.mean((val_pred > 0.5) == y_val)
    val_accs.append(val_acc)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d} | Loss: {train_losses[-1]:.4f} | Val Acc: {val_acc:.4f}")

print(f"\nFinal Val Accuracy: {val_accs[-1]:.4f}")

In [ ]:
# Plot training
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, color='#FF6B6B', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(val_accs, color='#4ECDC4', linewidth=2, marker='o')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. LoRA Configurations Summary

| Config | Rank | Alpha | Targets | Trainable % | Use Case |
|--------|------|-------|---------|-------------|----------|
| Minimal | 1 | 2 | q, v | 0.4% | Quick experiments |
| Standard | 4 | 8 | q, v | 1.6% | Most tasks |
| High-capacity | 8 | 16 | q, v | 3.1% | Complex tasks |
| Full attention | 4 | 8 | q,k,v,o | 3.1% | When q,v not enough |
| Maximum | 16 | 32 | q,k,v,o | 12.5% | Hard tasks |

**Best Practices:**
- Start with `rank=8, alpha=16, targets=['q', 'v']`
- Increase rank if underfitting
- Add k, o projections if still underfitting
- `alpha = 2 * rank` is a good default

## 🎯 Exercises

1. **Rank Impact**: Test rank=1, 2, 4, 8, 16, 32. Plot performance vs parameters.
2. **Alpha Tuning**: Try different alpha values (alpha = rank, 2*rank, 4*rank).
3. **Target Ablations**: Compare ['q'], ['v'], ['q', 'v'], ['q', 'k', 'v', 'o'].
4. **Real Model**: Apply to a real pretrained model using `peft` library.
5. **DoRA**: Implement Weight-Decomposed LoRA (DoRA).
6. **LoRA+**: Implement LoRA with different learning rates for A and B.